# Assignment 2 -- Part 1: Chi-Square via RDDs

Reproduces the Assignment 1 pipeline (per-category top-75 unigrams ranked by
chi-square plus the joined dictionary) using the PySpark **RDD API** with
transformations and actions only.

Steps:
1. Load the reviews and broadcast the stopword list.
2. Tokenise each review (case-fold, split on the assignment delimiters,
   drop one-character tokens and stopwords, dedupe within a review).
3. Compute `N_c` (docs per category) and `N` (total docs).
4. Compute `N_tc` (docs in `c` containing `t`) and re-key by term.
5. Per term, compute chi-square for every `(term, category)` it appears in.
6. Per category, keep the top 75 terms by chi-square.
7. Write `output_rdd.txt` in the Assignment 1 format (one line per
   category, final line = sorted union of selected terms).

The chi-square formula matches Assignment 1:
$\chi^2 = \dfrac{N \, (AD - BC)^2}{(A+B)(C+D)(A+C)(B+D)}$ with
$A=N_{tc}$, $B=N_t-N_{tc}$, $C=N_c-N_{tc}$, $D=N-N_c-N_t+N_{tc}$.

In [1]:
from __future__ import annotations

import heapq
import json
import re
from operator import add
from pathlib import Path

from pyspark import SparkConf, SparkContext

## Configuration

Paths default to the local dev set; on the cluster point `INPUT_PATH` at the
HDFS reviews path (e.g. `hdfs:///user/dic/reviews_devset.json`).

In [2]:
REPO_ROOT = Path.cwd()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "pyproject.toml").exists():
    REPO_ROOT = REPO_ROOT.parent

INPUT_PATH = str(REPO_ROOT / "data" / "reviews_devset.json")
STOPWORDS_PATH = str(REPO_ROOT / "data" / "stopwords.txt")
OUTPUT_PATH = str(REPO_ROOT / "output_rdd.txt")
TOP_N = 75

INPUT_PATH, STOPWORDS_PATH, OUTPUT_PATH

('/Users/martinweber/Development/Personal/data-intensive-computing/data/reviews_devset.json',
 '/Users/martinweber/Development/Personal/data-intensive-computing/data/stopwords.txt',
 '/Users/martinweber/Development/Personal/data-intensive-computing/output_rdd.txt')

In [3]:
conf = SparkConf().setAppName("assignment2-part1-rdd")
if not conf.contains("spark.master"):
    conf = conf.setMaster("local[*]")
sc = SparkContext.getOrCreate(conf=conf)
sc.setLogLevel("WARN")
sc

26/05/08 11:01:12 WARN Utils: Your hostname, MacBook-Pro-von-Martin-2.local resolves to a loopback address: 127.0.0.1; using 10.0.0.3 instead (on interface en0)
26/05/08 11:01:12 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/08 11:01:13 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


<SparkContext master=local[*] appName=assignment2-part1-rdd>

## Tokenisation

Same delimiter regex and rules as Assignment 1 so the chi-square scores are
directly comparable.

In [4]:
DELIMITER_PATTERN = re.compile(r"[\s\d()\[\]{}.!?,;:+=_\"'`~#@&*%\u20ac$\u00a7\\/-]+")

with open(STOPWORDS_PATH, encoding="utf-8") as fh:
    stopwords_set = set(fh.read().splitlines())
stopwords_bc = sc.broadcast(stopwords_set)

def tokenise(text: str, stopwords: set) -> set:
    folded = text.casefold()
    tokens = set()
    for token in DELIMITER_PATTERN.split(folded):
        if len(token) <= 1 or token in stopwords:
            continue
        tokens.add(token)
    return tokens

## Load reviews

Each line is one JSON review. We keep `(category, tokenset)` pairs and cache
the result -- it is reused for the per-category counts and the term/category
co-occurrence counts. Reviews missing `reviewText` keep an empty token set so
they still contribute to `N_c` (matching Assignment 1, where doc counts only
filter on `category`).

In [5]:
def parse_review(line: str):
    try:
        review = json.loads(line)
    except json.JSONDecodeError:
        return None
    category = review.get("category")
    if not category:
        return None
    text = review.get("reviewText") or ""
    tokens = tokenise(text, stopwords_bc.value) if text else set()
    return (category, tokens)

reviews = (
    sc.textFile(INPUT_PATH)
      .map(parse_review)
      .filter(lambda r: r is not None)
)
reviews.cache()
reviews.count()

78829

## `N_c` and `N`

Document counts per category and the corpus total, collected to the driver
and broadcast so later transformations can read them in O(1).

In [6]:
category_counts = (
    reviews.map(lambda kv: (kv[0], 1))
           .reduceByKey(add)
           .collectAsMap()
)
N = sum(category_counts.values())
N_c_bc = sc.broadcast(dict(category_counts))
len(category_counts), N

(22, 78829)

## `N_tc` -- per (term, category) document frequency

Emit `((term, category), 1)` once per *review* (tokens were deduplicated in
`parse_review`) and sum with `reduceByKey`.

In [7]:
term_cat_counts = (
    reviews.flatMap(lambda kv: (((t, kv[0]), 1) for t in kv[1]))
           .reduceByKey(add)
)
term_cat_counts.take(3)

[(('yum', 'Patio_Lawn_and_Garde'), 1),
 (('gift', 'Patio_Lawn_and_Garde'), 21),
 (('cuisine', 'Patio_Lawn_and_Garde'), 1)]

## Chi-square per (term, category)

Re-key by term so a single record carries every category in which a term
appears, plus its `N_t = sum N_tc`. Emit `(category, (chi2, term))` so the
next step can rank within each category.

In [8]:
def chi_square_for_term(term_and_pairs):
    term, pairs = term_and_pairs
    pairs = list(pairs)
    N_t = sum(n for _, n in pairs)
    if N_t == 0:
        return
    N_c_map = N_c_bc.value
    for category, N_tc in pairs:
        N_c = N_c_map.get(category)
        if N_c is None:
            continue
        A = N_tc
        B = N_t - N_tc
        C = N_c - N_tc
        D = N - N_c - N_t + N_tc
        denom = (A + B) * (C + D) * (A + C) * (B + D)
        if denom == 0:
            continue
        chi2 = N * (A * D - B * C) ** 2 / denom
        yield category, (chi2, term)

chi_pairs = (
    term_cat_counts.map(lambda kv: (kv[0][0], (kv[0][1], kv[1])))
                   .groupByKey()
                   .flatMap(chi_square_for_term)
)

## Top-N per category

`aggregateByKey` with a bounded heap keeps memory at `top_n` items per
category instead of materialising the full per-category list.

In [9]:
def heap_seq(acc, value):
    if len(acc) < TOP_N:
        heapq.heappush(acc, value)
    else:
        heapq.heappushpop(acc, value)
    return acc

def heap_comb(a, b):
    for v in b:
        a = heap_seq(a, v)
    return a

top_per_category = (
    chi_pairs.aggregateByKey([], heap_seq, heap_comb)
             .mapValues(lambda heap: sorted(heap, key=lambda p: -p[0]))
             .collect()
)
len(top_per_category)

22

## Write `output_rdd.txt`

One line per category (alphabetical) -- `<category> term:chi2 term:chi2 ...` --
and a final line with the sorted union of all selected terms.

In [10]:
results = {category: pairs for category, pairs in top_per_category}

lines = []
all_terms = set()
for category in sorted(results):
    pairs = results[category]
    tokens = " ".join(f"{term}:{chi2:.4f}" for chi2, term in pairs)
    lines.append(f"{category} {tokens}")
    all_terms.update(term for _, term in pairs)
lines.append(" ".join(sorted(all_terms)))

Path(OUTPUT_PATH).write_text("\n".join(lines) + "\n", encoding="utf-8")
print(f"Wrote {OUTPUT_PATH} ({len(results)} categories, {len(all_terms)} unique terms).")
print("\nFirst category preview:\n")
print(lines[0][:500])

Wrote /Users/martinweber/Development/Personal/data-intensive-computing/output_rdd.txt (22 categories, 1464 unique terms).

First category preview:

Apps_for_Android games:3081.1493 play:2158.3694 graphics:1505.5109 kindle:1470.8209 addictive:1311.9056 challenging:1038.1285 coins:1002.6648 addicting:990.8441 fire:956.1470 levels:825.3813 playing:692.9340 ads:642.3970 puzzles:596.7717 apps:548.7811 free:500.9885 bingo:409.2358 mahjong:322.0089 download:303.8649 faotd:288.8577 facebook:282.5171 downloaded:262.7702 hints:242.6103 solitaire:211.6430 android:211.5811 puzzle:198.8558 gameplay:198.5123 freezes:189.6774 unlock:185.7521 played:180.39


In [11]:
reviews.unpersist()
sc.stop()